# 5.2 · 构建客户端与 LLM 工具调用循环

**Building a client and the LLM tool-use loop**

> **学完本课你能：** 构建一个 MCP 客户端，让它连接服务器、把服务器的工具交给 LLM，并运行一个健壮的多轮智能体循环，具备错误回传、超时和迭代上限保护。

官方阅读材料：[Build an MCP client](https://modelcontextprotocol.io/docs/2026-07-28/develop/build-client)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 一个供驱动的多工具服务器
我们的服务器暴露了三个可以串联的工具（Tool）：`find_city_code` → `get_forecast` → `convert_temperature`。`get_forecast` 在遇到未知代码时会抛出 `ToolError`，这是一种刻意设计的、可读的失败，模型可以从中恢复。

In [ ]:
import json
import logging
import os
from types import SimpleNamespace

from mcp import Client
from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError

logging.getLogger("mcp").setLevel(logging.CRITICAL)

CITIES = {"paris": "PAR", "london": "LON", "tokyo": "TYO"}
FORECASTS = {"PAR": {"today": {"celsius": 17, "rain_probability": 80}, "tomorrow": {"celsius": 21, "rain_probability": 10}},
             "LON": {"today": {"celsius": 14, "rain_probability": 95}, "tomorrow": {"celsius": 15, "rain_probability": 60}},
             "TYO": {"today": {"celsius": 26, "rain_probability": 5}, "tomorrow": {"celsius": 24, "rain_probability": 30}}}

weather = MCPServer("weather")


@weather.tool()
def find_city_code(city: str) -> str:
    """Return the 3-letter forecast code for a city name, e.g. 'Paris' -> 'PAR'."""
    try:
        return CITIES[city.strip().lower()]
    except KeyError:
        raise ToolError(f"Unknown city {city!r}. Known: {sorted(CITIES)}")


@weather.tool()
def get_forecast(code: str, day: str = "today") -> dict:
    """Forecast for a city CODE (use find_city_code first) and day ('today' or 'tomorrow')."""
    if code not in FORECASTS:
        raise ToolError(f"Unknown city code {code!r}. Call find_city_code to get a valid code.")
    return {"code": code, "day": day, **FORECASTS[code][day]}


@weather.tool()
def convert_temperature(celsius: float) -> float:
    """Convert Celsius to Fahrenheit."""
    return round(celsius * 9 / 5 + 32, 1)


print("server ready:", weather.name)

### 2. 把 MCP 工具转换为模型的工具格式
这就是教程中的那个推导式，外加分页。注意哪些内容<em>没有</em>被转发：`outputSchema` 和注解（Annotations）留在宿主（Host）这边。

In [ ]:
async def list_all_tools(client):
    tools, cursor = [], None
    while True:
        page = await client.list_tools(cursor=cursor)
        tools += page.tools
        cursor = page.next_cursor
        if cursor is None:
            return tools


def to_anthropic(tools):
    return [{"name": t.name, "description": t.description, "input_schema": t.input_schema} for t in tools]


async with Client(weather) as client:
    mcp_tools = await list_all_tools(client)
    print("MCP tool as listed   :", json.dumps(mcp_tools[1].model_dump(by_alias=True, exclude_none=True))[:400], "...\n")
    print("Anthropic tool format:", json.dumps(to_anthropic(mcp_tools)[1], indent=2))

### 3. 一个与 Anthropic SDK 接口相同的确定性“模拟 LLM”
为了离线运行，我们需要一个与 Anthropic 客户端接口相同的东西：`llm.messages.create(model=..., max_tokens=..., tools=..., messages=...)`，返回一个带有 `stop_reason` 和 `content` 块（`type`、`text`、`id`、`name`、`input`）的对象。<em>策略</em>函数读取对话（包括之前的工具结果和 `is_error` 标志），并决定下一步行动，相当于一个脚本化规划器（Scripted Planner）。这让循环的行为变得可见且可重复。之后换成真正的 `anthropic.Anthropic()` 时，<strong>循环无需任何改动</strong>。

In [ ]:
def text_block(text):
    return SimpleNamespace(type="text", text=text)


_ids = iter(range(1, 10_000))
def tool_use(name, args):
    return SimpleNamespace(type="tool_use", id=f"toolu_{next(_ids):03d}", name=name, input=args)


def tool_trace(messages):
    """Pair every tool_use the 'model' made with its tool_result: [{name, input, content, is_error}, ...]."""
    calls = {}
    for m in messages:
        for block in (m["content"] if isinstance(m["content"], list) else []):
            if getattr(block, "type", None) == "tool_use":
                calls[block.id] = {"name": block.name, "input": block.input}
            elif isinstance(block, dict) and block.get("type") == "tool_result":
                calls[block["tool_use_id"]].update(content=block["content"], is_error=block["is_error"])
    return list(calls.values())


class FakeLLM:
    """Mimics anthropic.Anthropic().messages.create with a scripted policy(messages, tool_names) -> blocks."""

    def __init__(self, policy):
        self.policy, self.calls = policy, 0
        self.messages = self                              # so `llm.messages.create(...)` works

    def create(self, *, model, max_tokens, tools, messages):
        self.calls += 1
        blocks = self.policy(messages, {t["name"] for t in tools})
        stop = "tool_use" if any(b.type == "tool_use" for b in blocks) else "end_turn"
        return SimpleNamespace(stop_reason=stop, content=blocks)

下面是针对 <em>“Do I need an umbrella in Paris?”</em> 这类问题的脚本化策略。为了展示<strong>恢复</strong>过程，它首先<em>猜测</em>一个代码（“PARIS”），看到 `is_error` 结果后再纠正方向。随后工具 A 的输出（`PAR`）会作为工具 B 的输入。

In [ ]:
import re


def umbrella_policy(messages, tool_names):
    question = messages[0]["content"] if isinstance(messages[0]["content"], str) else ""
    city = re.search(r"in (\w+)", question).group(1)
    day = "tomorrow" if "tomorrow" in question else "today"
    trace = tool_trace(messages)
    if not trace:                                                    # round 1: a careless guess
        return [text_block(f"Let me check the forecast for {city}."), tool_use("get_forecast", {"code": city.upper(), "day": day})]
    last = trace[-1]
    if last["is_error"] and last["name"] == "get_forecast":         # recover: read the error, look up the code
        return [text_block("That code was wrong; looking it up."), tool_use("find_city_code", {"city": city})]
    if last["name"] == "find_city_code" and not last["is_error"]:   # chain: A's output becomes B's input
        return [tool_use("get_forecast", {"code": last["content"], "day": day})]
    if last["name"] == "get_forecast":
        f = json.loads(last["content"])
        verdict = "Yes, take an umbrella" if f["rain_probability"] >= 50 else "No umbrella needed"
        return [text_block(f"{verdict}: {f['rain_probability']}% chance of rain in {city} {day}, {f['celsius']}°C.")]
    return [text_block(f"I could not get a forecast: {last['content']}")]


print("policy defined")

### 4. 智能体循环
这是 Messages API 标准的手动工具调用循环，即智能体循环（Agent Loop）：当 `stop_reason == "tool_use"` 时持续循环，原样追加助手内容，然后追加<strong>一条</strong>包含<strong>所有</strong>工具结果的用户消息。它还加入了迭代上限保护（Iteration Guard），并把传输错误（例如超时）转换为 `is_error` 结果，让模型能够作出反应。

In [ ]:
from mcp.shared.exceptions import MCPError
import mcp_types as types

MODEL = "claude-opus-5-5"


def text_of(result: types.CallToolResult) -> str:
    return "\n".join(b.text for b in result.content if isinstance(b, types.TextContent))


async def run_agent(client, llm, messages, *, max_iterations=6, verbose=True):
    """Run one user turn to completion. Mutates and returns `messages` (the conversation state)."""
    tools = to_anthropic(await list_all_tools(client))
    for step in range(1, max_iterations + 1):
        response = llm.messages.create(model=MODEL, max_tokens=16000, tools=tools, messages=messages)
        messages.append({"role": "assistant", "content": response.content})
        for block in response.content:
            if block.type == "text" and verbose:
                print(f"  [{step}] model: {block.text}")
        if response.stop_reason != "tool_use":
            return messages
        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            try:
                result = await client.call_tool(block.name, block.input)
                content, is_error = text_of(result), bool(result.is_error)
            except MCPError as err:                                   # protocol/transport failure, e.g. timeout
                content, is_error = f"MCP error {err.error.code}: {err.error.message}", True
            if verbose:
                flag = "ERROR " if is_error else ""
                print(f"  [{step}] tool  {block.name}({json.dumps(block.input)}) -> {flag}{' '.join(content.split())[:90]}")
            tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": content, "is_error": is_error})
        messages.append({"role": "user", "content": tool_results})
    messages.append({"role": "assistant", "content": [text_block(f"Stopped: reached the {max_iterations}-step limit.")]})
    if verbose:
        print(f"  [guard] stopped after {max_iterations} iterations")
    return messages


def final_text(messages):
    return " ".join(b.text for b in messages[-1]["content"] if getattr(b, "type", None) == "text")


async with Client(weather) as client:
    llm = FakeLLM(umbrella_policy)
    convo = await run_agent(client, llm, [{"role": "user", "content": "Do I need an umbrella in Paris?"}])
    print("\nFINAL:", final_text(convo), f"| model calls: {llm.calls}, messages in history: {len(convo)}")

阅读这段输出轨迹。第 1 轮失败（`is_error`），第 2 轮通过 `find_city_code` 恢复，第 3 轮把它的输出 `PAR` 串联进 `get_forecast`，第 4 轮给出回答。教程中固定两次调用的版本在失败之后就会停下。

### 5. 检查会话状态
`messages` 列表<em>就是</em>智能体的记忆，也就是会话状态（Conversation State）。下面以 JSON 形式展示它，可以看到每个 `tool_use` id 与 `tool_result` 的配对。

In [ ]:
def jsonable(message):
    content = message["content"]
    if isinstance(content, list):
        content = [vars(b) if isinstance(b, SimpleNamespace) else b for b in content]
    return {"role": message["role"], "content": content}


for m in convo[:4]:
    print(json.dumps(jsonable(m))[:260])
print("...")

后续问题会被追加到<strong>同一个</strong>列表中。这就是跨轮次的会话状态。“there（那里）”之所以有意义，只是因为第一个问题（关于巴黎）还在历史记录中。我们的脚本化策略从 `messages[0]` 中读取城市，而真正的模型会直接阅读全部内容。

In [ ]:
def followup_policy(messages, tool_names):
    question = messages[-1]["content"] if isinstance(messages[-1]["content"], str) else None
    if question and "tomorrow" in question:                     # the model resolves "there" from history
        city = re.search(r"in (\w+)", messages[0]["content"]).group(1)
        return [tool_use("find_city_code", {"city": city})]
    last = tool_trace(messages)[-1]
    if last["name"] == "find_city_code":
        return [tool_use("get_forecast", {"code": last["content"], "day": "tomorrow"})]
    f = json.loads(last["content"])
    return [text_block(f"Tomorrow there: {f['rain_probability']}% rain, {f['celsius']}°C.")]


async with Client(weather) as client:
    convo.append({"role": "user", "content": "And tomorrow there?"})
    convo = await run_agent(client, FakeLLM(followup_policy), convo)
    print("FINAL:", final_text(convo), "| history length:", len(convo))

### 6. 制造故障：一个永不停止的模型
有缺陷的模型（或者被提示注入（Prompt Injection）的模型）可能会不停地调用工具。迭代上限保护（Iteration Guard）正是保住你预算的东西。

In [ ]:
def stubborn_policy(messages, tool_names):
    return [tool_use("convert_temperature", {"celsius": len(messages)})]   # always asks for another tool


async with Client(weather) as client:
    llm = FakeLLM(stubborn_policy)
    convo2 = await run_agent(client, llm, [{"role": "user", "content": "Loop forever"}], max_iterations=3)
    print("FINAL:", final_text(convo2), "| model calls:", llm.calls)

### 7. 在真实的 stdio 子进程上运行同一个循环
现在服务器作为一个独立进程运行，就像教程中那样。我们使用 `Client(stdio_client(StdioServerParameters(...)))`，这正是教程中的写法，即标准输入输出传输（stdio Transport）。服务器增加了一个刻意设计得很慢的工具。在 `read_timeout_seconds=1` 的设置下，这个慢调用会抛出超时 `MCPError`，循环把它转换为 `is_error` 结果，模型随后改用另一个工具。

In [ ]:
import sys
import tempfile
import textwrap
from pathlib import Path

from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client

workdir = Path(tempfile.mkdtemp())
server_file = workdir / "weather_server.py"
server_file.write_text(textwrap.dedent('''
    import time
    from mcp.server import MCPServer
    from mcp.server.mcpserver.exceptions import ToolError

    server = MCPServer("weather-stdio", log_level="CRITICAL")
    CODES = {"paris": "PAR", "london": "LON"}

    @server.tool()
    def find_city_code(city: str) -> str:
        """Return the 3-letter forecast code for a city name."""
        if city.lower() not in CODES:
            raise ToolError(f"Unknown city {city!r}")
        return CODES[city.lower()]

    @server.tool()
    def live_radar(code: str) -> str:
        """High-resolution live radar (slow)."""
        time.sleep(3)
        return f"radar for {code}: heavy rain"

    @server.tool()
    def get_forecast(code: str) -> dict:
        """Forecast for a city CODE."""
        return {"code": code, "rain_probability": {"PAR": 80, "LON": 95}.get(code, 0)}

    if __name__ == "__main__":
        server.run(transport="stdio")
'''))


def radar_policy(messages, tool_names):
    trace = tool_trace(messages)
    if not trace:
        return [tool_use("find_city_code", {"city": "London"})]
    last = trace[-1]
    if last["name"] == "find_city_code":
        return [tool_use("live_radar", {"code": last["content"]})]
    if last["name"] == "live_radar" and last["is_error"]:
        return [text_block("Radar timed out; using the forecast instead."), tool_use("get_forecast", {"code": "LON"})]
    f = json.loads(last["content"])
    return [text_block(f"Yes: {f['rain_probability']}% chance of rain in London (from {last['name']}).")]


params = StdioServerParameters(command=sys.executable, args=[str(server_file)])
async with Client(stdio_client(params), read_timeout_seconds=1.0) as client:
    print("connected over stdio:", client.server_info.name, "| protocol", client.protocol_version)
    print("tools:", [t.name for t in await list_all_tools(client)])
    convo3 = await run_agent(client, FakeLLM(radar_policy), [{"role": "user", "content": "Is it raining in London?"}])
    print("FINAL:", final_text(convo3))
print("subprocess stopped when the async with block exited")

### 8. <em>不是</em>工具错误的错误
在这个 SDK 中，调用一个不存在的工具或传入无效参数，仍然会返回一个带 `isError: true` 的结果。而超时过长的调用则会抛出异常。一个好的宿主（Host）能处理全部三种情况，而不会让聊天崩溃。

In [ ]:
async with Client(StdioServerParameters(command=sys.executable, args=[str(server_file)])) as client:
    r = await client.call_tool("no_such_tool", {})
    print("unknown tool  -> isError:", r.is_error, "|", text_of(r))
    r = await client.call_tool("find_city_code", {"city": "Atlantis"})
    print("ToolError     -> isError:", r.is_error, "|", text_of(r))
    try:
        await client.call_tool("live_radar", {"code": "PAR"}, read_timeout_seconds=0.5)
    except MCPError as err:
        print("timeout       -> MCPError", err.error.code, err.error.message)

### 9. 可选：真正的 Claude 循环
如果设置了 `ANTHROPIC_API_KEY`，完全相同的 `run_agent` 就会使用 `anthropic.Anthropic()` 运行。按照教程的做法，你通常会从一个被 git 忽略的 `.env` 中执行 `load_dotenv()`。Claude 通常会自己先调用 `find_city_code`，因为工具描述就是这样告诉它的。

In [ ]:
if os.environ.get("ANTHROPIC_API_KEY"):
    import anthropic

    claude = anthropic.Anthropic()            # reads ANTHROPIC_API_KEY from the environment
    async with Client(weather) as client:
        convo4 = await run_agent(client, claude, [{"role": "user", "content":
            "Do I need an umbrella in London tomorrow? Also give the temperature in Fahrenheit."}])
        print("FINAL:", final_text(convo4))
else:
    print("skipped: set ANTHROPIC_API_KEY to run the same loop with claude-opus-5-5 "
          "(expected: find_city_code -> get_forecast -> convert_temperature -> answer).")

### 动手练习
1. 为 <em>“Is it warmer in Tokyo or London today, in Fahrenheit?”</em> 编写一个策略，让它<strong>在一个轮次中请求两个 `tool_use` 块</strong>（并行调用），然后再调用 `convert_temperature`。检查你的循环是否在一条用户消息中发送了这两个结果。
2. 给 `run_agent` 添加一个<strong>同意关卡</strong>：在调用任何名称以 `delete_` 或 `send_` 开头的工具之前，先调用一个 `approve(name, args)` 函数；如果它返回 `False`，就回传带有 “User denied this action” 的 `is_error: True`。用一个模拟的破坏性工具来测试它。
3. 让 `run_agent` 在把工具展示给模型时，将其命名空间化为 `weather__get_forecast`，并在 `call_tool` 之前去掉前缀。为什么在有两个服务器时这一点很重要？
4. 删除 `run_agent` 中的 `try/except MCPError`，然后重新运行第 7 节。对话会发生什么？为什么按工具调用捕获错误比按查询捕获更好？